# localrag demo: accurate, cited answers from a 0.8B local model

Hybrid retrieval (BM25 + dense, RRF fusion, cross-encoder rerank) feeding a grammar-constrained 0.8B generator whose every claim is verified against the passage it cites. By default the demo uses the recommended configuration: NLI verification plus the `dpo-refusal` adapter, which answers what the documents cover and refuses what they don't.

Every technique is a parameter in the next cell. The last sections compare the base model with each fine-tuned adapter and show the evaluation table.

## Parameters
Every technique is a switch. `None` keeps the default from `configs/default.yaml`.

In [1]:
# --- model
GENERATOR = "qwen3.5-0.8b"        # "qwen3.5-0.8b" | "qwen3-0.6b"
ANSWER_ADAPTER = "dpo-refusal"    # None | "dpo" | "dpo-refusal" | "grpo"  (python -m localrag train <name>;
                                  #  falls back to the base model if the adapter isn't trained)
DECOMPOSE_ADAPTER = None          # None | "grpo-decompose"         (needs: python -m localrag train grpo-decompose)

# --- ingestion (changing these requires re-running the ingest cell)
PARSER = "pymupdf"                # "pymupdf" (hand-written) | "docling"
CHILD_SIZE = 0                    # >0: parent-child retrieval with children of this many characters
CONTEXTUALIZE = False             # LLM-written context prepended to every chunk before indexing

# --- retrieval
RETRIEVAL_MODE = "hybrid"         # "dense" | "sparse" | "hybrid"
SPARSE = "bm25"                   # "bm25" | "splade"
DENSE_BACKEND = "bi_encoder"      # "bi_encoder" (one vector per chunk) | "late_interaction" (ColBERT MaxSim)
RERANKER = "cross_encoder"        # "cross_encoder" (bge-reranker-v2-m3) | "llama" (Qwen3-Reranker) | None
TOP_K = 5                         # passages given to the LLM (use ~3 with CONTEXT="section")

# --- agent
DECOMPOSE = True
RETRIEVAL_STRATEGY = "single"     # "single" | "iterative" (LLM names what is missing and searches again)
CRAG = False                      # corrective RAG: rewrite + re-retrieve when evidence is weak
CONTEXT = "chunk"                 # "chunk" | "window" (+ neighbouring chunks) | "section" (retrieve less, read more)
COMPRESSION = "none"              # "none" | "sentences" (keep the most query-relevant sentences per passage)
ANSWER_MODE = "grounded"          # "grounded" | "grounded_evidence" | "freetext" (original prompt-only citations)
VERIFIER = "nli"                  # "none" | "reranker" | "nli"
SELF_CORRECT = False              # regenerate once if most claims are unsupported
MIN_RELEVANCE = 0.0               # refuse below this top rerank score (python -m localrag calibrate)

In [2]:
import os
import warnings
from pathlib import Path

os.environ["TQDM_DISABLE"] = "1"               # no progress bars in the saved notebook
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")

from IPython.display import Markdown, display

from localrag.config import load_config
from localrag.pipeline import RAGPipeline, ingest

MODELS = {"qwen3.5-0.8b": ("unsloth/Qwen3.5-0.8B-GGUF", "Qwen3.5-0.8B-Q8_0.gguf"),
          "qwen3-0.6b": ("Qwen/Qwen3-0.6B-GGUF", "Qwen3-0.6B-Q8_0.gguf")}
adapters = {}
for step, name in (("answer", ANSWER_ADAPTER), ("decompose", DECOMPOSE_ADAPTER)):
    if name and Path(f"adapters/{name}.gguf").exists():
        adapters[step] = f"adapters/{name}.gguf"
    elif name:
        print(f"adapters/{name}.gguf not found (python -m localrag train {name}); using the base model for '{step}'")

overrides = {
    "llm.repo_id": MODELS[GENERATOR][0], "llm.filename": MODELS[GENERATOR][1], "llm.adapters": adapters,
    "ingest.parser": PARSER, "ingest.child_size": CHILD_SIZE, "ingest.contextualize": CONTEXTUALIZE,
    "retrieval.mode": RETRIEVAL_MODE, "retrieval.sparse": SPARSE, "retrieval.dense_backend": DENSE_BACKEND,
    "retrieval.top_k": TOP_K, "agent.retrieval_strategy": RETRIEVAL_STRATEGY, "agent.context": CONTEXT,
    "agent.compression": COMPRESSION,
    "agent.decompose": DECOMPOSE, "agent.crag": CRAG, "agent.answer_mode": ANSWER_MODE,
    "agent.verifier": VERIFIER, "agent.self_correct": SELF_CORRECT, "agent.min_relevance": MIN_RELEVANCE,
    "agent.support_threshold": 0.5 if VERIFIER == "nli" else 0.05,
}
if RERANKER is None:
    overrides["retrieval.reranker_model"] = None
else:
    overrides["retrieval.reranker_backend"] = RERANKER
# non-default ingestion settings get their own index directory
if (PARSER, CHILD_SIZE, CONTEXTUALIZE) != ("pymupdf", 0, False):
    overrides["index_dir"] = f"index/{PARSER}_c{CHILD_SIZE}_ctx{int(CONTEXTUALIZE)}"

cfg = load_config("configs/default.yaml", overrides)
cfg.index_path

PosixPath('index')

## Build the index
Only needed once per ingestion setting, or after the PDFs in `documents/` change.

In [3]:
if not (cfg.index_path / "chunks.jsonl").exists():
    ingest(cfg)

## Ask questions
`RAGPipeline` starts a `llama-server` for the GGUF model (plus adapters, if any).

In [4]:
rag = RAGPipeline(cfg)

def ask(q, trace=True):
    ans = rag.answer(q)
    if trace:
        print("action:", ans.action, "| subqueries:", ans.subqueries, "| timings:", ans.timings,
              "| trace:", ans.trace)
        for i, h in enumerate(ans.passages, 1):
            print(f"[{i}] {h.score:.3f} {h.chunk.source} p.{h.chunk.page} | {h.chunk.section[:70]}")
        for c in ans.claims:
            print(f"   supported={c.supported} ({(c.support_score or 0):.2f}) {c.text[:90]} {c.citations}")
    display(Markdown(ans.text))
    return ans

In [5]:
ask("Hello!", trace=False);

Hello! How can I assist you today? 😊

In [6]:
ask("What must design verification confirm under the quality system regulation?");

action: SEARCH | subqueries: ['What does design verification confirm according to quality system regulations?', 'What is the requirement for design verification in quality system compliance?'] | timings: {'route': 0.068, 'decompose': 0.127, 'retrieve': 9.009, 'generate': 0.315, 'verify': 2.991} | trace: {}
[1] 0.907 fda_general_principles_software_validation.pdf p.10 | Industry and FDA Staff > General Principles of Software Validation > S
[2] 0.870 fda_general_principles_software_validation.pdf p.11 | Industry and FDA Staff > General Principles of Software Validation > S
[3] 0.824 fda_general_principles_software_validation.pdf p.15 | Industry and FDA Staff > General Principles of Software Validation > S
[4] 0.754 21cfr820_quality_system_regulation.pdf p.7 | Subpart C—Design Controls
[5] 0.731 fda_general_principles_software_validation.pdf p.11 | Industry and FDA Staff > General Principles of Software Validation > S
   supported=True (1.00) Design verification confirms that the design o

Design verification confirms that the design output meets the design input requirements. [2] [4] [5] The Quality System regulation requires design verification procedures to confirm this conformity. [2] [4] [5]

[2] fda_general_principles_software_validation.pdf, p. 11 — Industry and FDA Staff > General Principles of Software Validation > SECTION 2. SCOPE > SECTION 3. CONTEXT FOR SOFTWARE VALIDATION > 3.1.2 Verification and Validation
[4] 21cfr820_quality_system_regulation.pdf, p. 7 — Subpart C—Design Controls
[5] fda_general_principles_software_validation.pdf, p. 11 — Industry and FDA Staff > General Principles of Software Validation > SECTION 2. SCOPE > SECTION 3. CONTEXT FOR SOFTWARE VALIDATION > 3.1.2 Verification and Validation

A comparison across two documents (multi-hop):

In [7]:
ask("How do the off-the-shelf software guidance and the cybersecurity guidance differ in what manufacturers must document about third-party software?");

action: SEARCH | subqueries: ['How do off-the-shelf software guidance and cybersecurity guidance differ in what manufacturers must document about third-party software?', 'What documentation requirements do manufacturers have for third-party software under off-the-shelf guidance versus cybersecurity guidance?', 'What specific documentation must manufacturers include regarding third-party software when using off-the-shelf guidance versus cybersecurity guidance?'] | timings: {'route': 0.077, 'decompose': 0.186, 'retrieve': 0.075, 'generate': 0.791, 'verify': 0.244} | trace: {}
[1] 0.976 fda_cybersecurity_in_medical_devices.pdf p.19 | Guidance for Industry and Food and Drug Administration Staff > V. Usin
[2] 0.732 fda_cybersecurity_in_medical_devices.pdf p.21 | Guidance for Industry and Food and Drug Administration Staff > V. Usin
[3] 0.701 fda_cybersecurity_in_medical_devices.pdf p.40 | Guidance for Industry and Food and Drug Administration Staff > VII. Cy
[4] 0.569 fda_general_principles

Manufacturers should document cybersecurity risks and controls associated with third-party software components. [1] [3] The cybersecurity guidance requires manufacturers to assess cybersecurity risk for third-party software components. [1] [3] The cybersecurity guidance requires manufacturers to document all software components of a device. [1] [3] The cybersecurity guidance requires addressing or mitigating risks associated with third-party software components. [1] [3] The cybersecurity guidance requires manufacturers to document cybersecurity risks and controls for third-party software. [1] [3] The cybersecurity guidance requires manufacturers to address or mitigate risks associated with third-party software components. [1] [3]

[1] fda_cybersecurity_in_medical_devices.pdf, p. 19 — Guidance for Industry and Food and Drug Administration Staff > V. Using an SPDF to Manage Cybersecurity Risks
[3] fda_cybersecurity_in_medical_devices.pdf, p. 40 — Guidance for Industry and Food and Drug Administration Staff > VII. Cyber Devices > D. Modifications

> **Known weak spot:** comparison questions. Here retrieval found only the cybersecurity side, so the answer covers one document and repeats itself. Multi-hop retrieval finds all evidence pages only about half the time (All@5 ≈ 0.5); see the evaluation table below.

A question the documents do not answer: the system should refuse rather than improvise.

In [8]:
ask("What are the EU MDR Annex VIII classification rules for software?");

action: SEARCH | subqueries: ['EU MDR Annex VIII classification rules for software', 'EU MDR Annex VIII software classification rules', 'EU MDR Annex VIII software classification rules 2024'] | timings: {'route': 0.077, 'decompose': 0.125, 'retrieve': 0.068, 'generate': 0.085, 'verify': 0.0} | trace: {}
[1] 0.041 fda_general_principles_software_validation.pdf p.8 | Industry and FDA Staff > General Principles of Software Validation > S
[2] 0.020 fda_off_the_shelf_software.pdf p.5 | Medical Devices Guidance for Industry and Food and Drug Administration
[3] 0.009 fda_cybersecurity_in_medical_devices.pdf p.48 | Guidance for Industry and Food and Drug Administration Staff > Appendi
[4] 0.007 fda_general_principles_software_validation.pdf p.5 | Industry and FDA Staff > Table of Contents > SECTION 4. PRINCIPLES OF 
[5] 0.005 fda_general_principles_software_validation.pdf p.4 | Industry and FDA Staff > Table of Contents > SECTION 4. PRINCIPLES OF 


The provided documents do not contain enough information to answer this question.

## Base vs. fine-tuned adapters
The same questions answered by the base model and each LoRA adapter (trained with `python -m localrag train dpo | dpo-refusal | grpo`). The last question is not covered by the documents: only `dpo-refusal` is trained to refuse it.

In [9]:
from localrag.llm import release_llms

variants = [None] + [n for n in ("dpo", "dpo-refusal", "grpo") if Path(f"adapters/{n}.gguf").exists()]
questions = [
    "What records must be kept for design reviews?",
    "How should a manufacturer validate off-the-shelf software used in a device?",
    "What are the EU MDR Annex VIII classification rules for software?",   # not in the documents
]
if len(variants) == 1:
    print("No adapters found; train with: python -m localrag train dpo  (or dpo-refusal, grpo)")
else:
    release_llms()
    answers = {}
    for name in variants:
        c = load_config("configs/default.yaml",
                        {**overrides, "llm.adapters": {"answer": f"adapters/{name}.gguf"} if name else {}})
        pipe = RAGPipeline(c)
        for q in questions:
            answers[q, name] = pipe.answer(q)
        release_llms()

    for q in questions:
        md = f"### {q}\n\n"
        for name in variants:
            a = answers[q, name]
            status = "refused" if a.insufficient_context else f"{sum(c.supported is not False for c in a.claims)} verified claims"
            md += f"**{name or 'base'}** · _{status}_\n\n{a.text}\n\n---\n\n"
        display(Markdown(md))

### What records must be kept for design reviews?

**base** · _5 verified claims_

Design reviews must be documented in the Design History File (DHF). [3] The DHF must contain the results of a design review, including the date and the individual(s) performing the review. [3] Design reviews must be conducted at appropriate stages of the device's design development. [3] Formal design reviews are especially important at or near the end of the requirements activity before major resources are committed to specific design solutions. [4] Problems found at this point can be resolved more easily, save time and money, and reduce the likelihood of missing a critical issue. [4]

[3] 21cfr820_quality_system_regulation.pdf, p. 7 — Subpart C—Design Controls
[4] fda_general_principles_software_validation.pdf, p. 15 — Industry and FDA Staff > General Principles of Software Validation > SECTION 2. SCOPE > SECTION 3. CONTEXT FOR SOFTWARE VALIDATION > 3.5 DESIGN REVIEW

---

**dpo** · _2 verified claims_

The DHF shall contain or reference the records necessary to demonstrate that the design was developed in accordance with the approved design plan and the requirements of this part. [2] Design review results, including identification of the design, the date, and the individual(s) performing the review, shall be documented in the DHF. [3]

[2] 21cfr820_quality_system_regulation.pdf, p. 8 — Subpart C—Design Controls
[3] 21cfr820_quality_system_regulation.pdf, p. 7 — Subpart C—Design Controls

---

**dpo-refusal** · _4 verified claims_

Design review results must be documented in the Design History File (DHF). [2] [3] Design verification results must be documented in the Design History File (DHF). [3] Records shall be legible and stored to minimize deterioration and to prevent loss. [5] Records stored in automated data processing systems shall be backed up. [5]

[2] 21cfr820_quality_system_regulation.pdf, p. 8 — Subpart C—Design Controls
[3] 21cfr820_quality_system_regulation.pdf, p. 7 — Subpart C—Design Controls
[5] 21cfr820_quality_system_regulation.pdf, p. 13 — Subpart M—Records

---

**grpo** · _1 verified claims_

Design review means a documented, comprehensive, systematic examination of a design to evaluate the adequacy of the design requirements, to evaluate the capability of the design to meet these requirements, and to identify problems. [1]

[1] 21cfr820_quality_system_regulation.pdf, p. 4 — Subpart A—General Provisions > § 820.1 Scope.

---



### How should a manufacturer validate off-the-shelf software used in a device?

**base** · _8 verified claims_

The device manufacturer should consider auditing the vendor's design and development methodologies used in the construction of the OTS software. [1] Such audits can be conducted by the device manufacturer or by a qualified third party. [1] The audit should demonstrate that the vendor's procedures for and results of the verification and validation activities performed the OTS software are appropriate and sufficient for the safety and effectiveness requirements of the medical device. [1] Some vendors who are not accustomed to operating in a regulated environment may not have a documented life cycle process that can support the device manufacturer's validation requirement. [2] For many applications black box testing alone is not sufficient. [2] Depending upon the risk of the device produced, the role of the OTS software in the process, the ability to audit the vendor, and the sufficiency of vendor-supplied information, the use of OTS software or equipment may or may not be appropriate. [2] The device manufacturer should also consider the implications (if any) for continued maintenance and support of the OTS software should the vendor terminate their support. [2] Most of the automated equipment and systems used by device manufacturers are supplied by third-party vendors and are purchased off-the-shelf (OTS). [3]

[1] fda_general_principles_software_validation.pdf, p. 38 — Industry and FDA Staff > General Principles of Software Validation > EQUIPMENT AND QUALITY SYSTEM SOFTWARE > 6.3. VALIDATION OF OFF-THE-SHELF SOFTWARE AND AUTOMATED EQUIPMENT
[2] fda_general_principles_software_validation.pdf, p. 38 — Industry and FDA Staff > General Principles of Software Validation > EQUIPMENT AND QUALITY SYSTEM SOFTWARE > 6.3. VALIDATION OF OFF-THE-SHELF SOFTWARE AND AUTOMATED EQUIPMENT
[3] fda_general_principles_software_validation.pdf, p. 38 — Industry and FDA Staff > General Principles of Software Validation > EQUIPMENT AND QUALITY SYSTEM SOFTWARE > 6.3. VALIDATION OF OFF-THE-SHELF SOFTWARE AND AUTOMATED EQUIPMENT

---

**dpo** · _8 verified claims_

The device manufacturer should consider auditing the vendor's design and development methodologies used in the construction of the OTS software. [1] [2] Some vendors who are not accustomed to operating in a regulated environment may not have a documented life cycle process that can support the device manufacturer's validation requirement. [2] Where necessary validation information is not available from the vendor, the device manufacturer will need to perform sufficient system level "black box" testing to establish that the software meets their "user needs and intended uses." [2] [3] Black box testing alone may not be sufficient for many applications, especially if there are suitable alternatives available. [2] The device manufacturer should also consider the implications (if any) for continued maintenance and support of the OTS software should the vendor terminate their support. [2] For some off-the-shelf software development tools, such as software compilers, linkers, editors, and operating systems, exhaustive black-box testing by the device manufacturer may be impractical. [5] However, their proper operation may be satisfactorily inferred by other means, such as compiler certification by independent third-party testing or available operational information from the vendor. [5] System-level validation testing of the application software should address all the operating system services used, including maximum loading conditions, file operations, and handling of system error. [5]

[1] fda_general_principles_software_validation.pdf, p. 38 — Industry and FDA Staff > General Principles of Software Validation > EQUIPMENT AND QUALITY SYSTEM SOFTWARE > 6.3. VALIDATION OF OFF-THE-SHELF SOFTWARE AND AUTOMATED EQUIPMENT
[2] fda_general_principles_software_validation.pdf, p. 38 — Industry and FDA Staff > General Principles of Software Validation > EQUIPMENT AND QUALITY SYSTEM SOFTWARE > 6.3. VALIDATION OF OFF-THE-SHELF SOFTWARE AND AUTOMATED EQUIPMENT
[3] fda_general_principles_software_validation.pdf, p. 38 — Industry and FDA Staff > General Principles of Software Validation > EQUIPMENT AND QUALITY SYSTEM SOFTWARE > 6.3. VALIDATION OF OFF-THE-SHELF SOFTWARE AND AUTOMATED EQUIPMENT
[5] fda_general_principles_software_validation.pdf, p. 38 — Industry and FDA Staff > General Principles of Software Validation > EQUIPMENT AND QUALITY SYSTEM SOFTWARE > 6.3. VALIDATION OF OFF-THE-SHELF SOFTWARE AND AUTOMATED EQUIPMENT

---

**dpo-refusal** · _8 verified claims_

The device manufacturer should consider auditing the vendor's design and development methodologies used in the construction of the OTS software. [1] [2] Some vendors who are not accustomed to operating in a regulated environment may not have a documented life cycle process that can support the device manufacturer's validation requirement. [2] Where necessary validation information is not available from the vendor, the device manufacturer will need to perform sufficient system level "black box" testing to establish that the software meets their "user needs and intended uses." [2] [3] For some off-the-shelf software development tools, such as software compilers, linkers, editors, and operating systems, exhaustive black-box testing may be impractical. [3] [5] However, their proper operation may be satisfactorily inferred by other means. [3] [5] For example, compilers are frequently certified by independent third-party testing, and commercial software products may have "bug lists", system requirements and other operational information available from the vendor that can be compared to the device manufacturer's intended use. [3] [5] Off-the-shelf operating systems need not be validated as a separate program. [3] [5] However, system-level validation testing of the application software should address all the operating system services used, including maximum loading conditions, file operations, handling of system error. [3] [5]

[1] fda_general_principles_software_validation.pdf, p. 38 — Industry and FDA Staff > General Principles of Software Validation > EQUIPMENT AND QUALITY SYSTEM SOFTWARE > 6.3. VALIDATION OF OFF-THE-SHELF SOFTWARE AND AUTOMATED EQUIPMENT
[2] fda_general_principles_software_validation.pdf, p. 38 — Industry and FDA Staff > General Principles of Software Validation > EQUIPMENT AND QUALITY SYSTEM SOFTWARE > 6.3. VALIDATION OF OFF-THE-SHELF SOFTWARE AND AUTOMATED EQUIPMENT
[3] fda_general_principles_software_validation.pdf, p. 38 — Industry and FDA Staff > General Principles of Software Validation > EQUIPMENT AND QUALITY SYSTEM SOFTWARE > 6.3. VALIDATION OF OFF-THE-SHELF SOFTWARE AND AUTOMATED EQUIPMENT
[5] fda_general_principles_software_validation.pdf, p. 38 — Industry and FDA Staff > General Principles of Software Validation > EQUIPMENT AND QUALITY SYSTEM SOFTWARE > 6.3. VALIDATION OF OFF-THE-SHELF SOFTWARE AND AUTOMATED EQUIPMENT

---

**grpo** · _3 verified claims_

Where possible and depending upon the device risk involved, the device manufacturer should consider auditing the vendor's design and development methodologies used in the construction of the OTS software and should assess the development and validation documentation generated for the OTS software. [1] Some vendors who are not accustomed to operating in a regulated environment may not have a documented life cycle process that can support the device manufacturer's validation requirement. Other vendors may not permit an audit. [2] For many applications black box testing alone is not sufficient. Depending upon the risk of the device produced, the role of the OTS software in the process, the ability to audit the vendor, and the sufficiency of vendor-supplied information, the use of OTS software or equipment may or may not be appropriate, especially if there are suitable alternatives available. [2]

[1] fda_general_principles_software_validation.pdf, p. 38 — Industry and FDA Staff > General Principles of Software Validation > EQUIPMENT AND QUALITY SYSTEM SOFTWARE > 6.3. VALIDATION OF OFF-THE-SHELF SOFTWARE AND AUTOMATED EQUIPMENT
[2] fda_general_principles_software_validation.pdf, p. 38 — Industry and FDA Staff > General Principles of Software Validation > EQUIPMENT AND QUALITY SYSTEM SOFTWARE > 6.3. VALIDATION OF OFF-THE-SHELF SOFTWARE AND AUTOMATED EQUIPMENT

---



### What are the EU MDR Annex VIII classification rules for software?

**base** · _refused_

The provided documents do not contain enough information to answer this question.

---

**dpo** · _2 verified claims_

The text mentions that medical device software is subject to design control provisions and validation requirements, but this does not define the EU MDR Annex VIII classification rules. [1] [2] [3] The text mentions that software used to automate device design, testing, component acceptance, manufacturing, labeling, packaging, distribution, complaint handling, or other aspects of the quality system must be validated, but this does not define EU MDR Annex VIII classification rules. [1] [2] [3]

[1] fda_general_principles_software_validation.pdf, p. 8 — Industry and FDA Staff > General Principles of Software Validation > SECTION 2. SCOPE > 2.4. REGULATORY REQUIREMENTS FOR SOFTWARE VALIDATION
[2] fda_off_the_shelf_software.pdf, p. 5 — Medical Devices Guidance for Industry and Food and Drug Administration Staff > II. Scope
[3] fda_cybersecurity_in_medical_devices.pdf, p. 48 — Guidance for Industry and Food and Drug Administration Staff > Appendix 1. Security Control Categories and Associated Recommendations > · Code Integrity

---

**dpo-refusal** · _refused_

The provided documents do not contain enough information to answer this question.

---

**grpo** · _refused_

The provided documents do not contain enough information to answer this question.

---



## Evaluation results
`python -m localrag eval` runs every variant in `configs/ablations.yaml` on the held-out test split; the table lands in `eval/results/summary.md`.

In [10]:
summary = Path("eval/results/summary.md")
display(Markdown(summary.read_text() if summary.exists() else "Run `python -m localrag eval` first."))

Split: test: 104 single-hop, 47 multi-hop, 17 unanswerable questions. ± = 95% bootstrap CI.

| variant | R@5 | multi_All@5 | correct | multi_correct | faithful | cited | cites_gold | multi_cites_all | false_refusal | unans_refusal | latency_s |
|---|---|---|---|---|---|---|---|---|---|---|---|
| v0_qwen3-0.6b | 0.952 ±0.038 | 0.468 ±0.149 | 0.812 ±0.065 | 0.585 ±0.085 | 0.116 ±0.036 | 0.151 | 0.221 | 0.064 | 0.0 | 0.0 | 0.62 |
| default | 0.942 ±0.043 | 0.489 ±0.149 | 0.779 ±0.055 | 0.617 ±0.09 | 0.901 ±0.03 | 1.0 | 0.923 | 0.468 | 0.0 | 0.059 | 0.98 |
| docling_parser | 0.942 ±0.043 | 0.468 ±0.149 | 0.803 ±0.055 | 0.543 ±0.085 | 0.886 ±0.032 | 1.0 | 0.875 | 0.426 | 0.0 | 0.118 | 0.92 |
| chunk_600 | 0.942 ±0.043 | 0.468 ±0.149 | 0.76 ±0.06 | 0.553 ±0.085 | 0.872 ±0.034 | 1.0 | 0.923 | 0.383 | 0.013 | 0.059 | 0.91 |
| parent_child | 0.952 ±0.038 | 0.553 ±0.149 | 0.774 ±0.053 | 0.574 ±0.101 | 0.863 ±0.034 | 1.0 | 0.913 | 0.489 | 0.0 | 0.118 | 1.01 |
| contextual | 0.971 ±0.034 | 0.553 ±0.149 | 0.764 ±0.058 | 0.617 ±0.101 | 0.875 ±0.037 | 1.0 | 0.942 | 0.489 | 0.0 | 0.118 | 1.0 |
| dense_only | 0.942 ±0.043 | 0.447 ±0.149 | 0.774 ±0.06 | 0.574 ±0.106 | 0.872 ±0.034 | 1.0 | 0.923 | 0.447 | 0.0 | 0.176 | 1.02 |
| bm25_only | 0.923 ±0.048 | 0.426 ±0.149 | 0.76 ±0.06 | 0.553 ±0.112 | 0.879 ±0.036 | 1.0 | 0.904 | 0.34 | 0.007 | 0.059 | 0.97 |
| splade_hybrid | 0.952 ±0.038 | 0.404 ±0.149 | 0.755 ±0.06 | 0.532 ±0.096 | 0.915 ±0.029 | 1.0 | 0.894 | 0.362 | 0.007 | 0.059 | 1.02 |
| qwen3_embedding | 0.962 ±0.038 | 0.532 ±0.149 | 0.788 ±0.053 | 0.585 ±0.106 | 0.895 ±0.032 | 1.0 | 0.913 | 0.426 | 0.0 | 0.059 | 1.01 |
| no_rerank | 0.923 ±0.048 | 0.298 ±0.138 | 0.76 ±0.053 | 0.521 ±0.101 | 0.864 ±0.034 | 1.0 | 0.885 | 0.255 | 0.0 | 0.059 | 0.85 |
| qwen3_reranker | 0.952 ±0.038 | 0.468 ±0.149 | 0.808 ±0.053 | 0.564 ±0.106 | 0.859 ±0.041 | 1.0 | 0.875 | 0.426 | 0.0 | 0.118 | 1.25 |
| no_decompose | 0.952 ±0.038 | 0.553 ±0.149 | 0.74 ±0.06 | 0.521 ±0.101 | 0.892 ±0.032 | 1.0 | 0.894 | 0.489 | 0.0 | 0.059 | 0.83 |
| freetext_citations | 0.942 ±0.043 | 0.489 ±0.149 | 0.856 ±0.055 | 0.713 ±0.101 | 0.043 ±0.026 | 0.051 | 0.058 | 0.043 | 0.0 | 0.0 | 0.81 |
| no_verifier | 0.942 ±0.043 | 0.489 ±0.149 | 0.788 ±0.053 | 0.617 ±0.101 | 0.899 ±0.034 | 1.0 | 0.904 | 0.447 | 0.0 | 0.059 | 0.99 |
| evidence_first | 0.942 ±0.043 | 0.489 ±0.149 | 0.769 ±0.053 | 0.532 ±0.101 | 0.887 ±0.04 | 1.0 | 0.904 | 0.34 | 0.007 | 0.059 | 1.15 |
| nli_verifier | 0.942 ±0.043 | 0.489 ±0.149 | 0.736 ±0.058 | 0.479 ±0.096 | 0.959 ±0.021 | 1.0 | 0.894 | 0.277 | 0.053 | 0.412 | 1.0 |
| self_correct | 0.942 ±0.043 | 0.489 ±0.149 | 0.726 ±0.065 | 0.511 ±0.096 | 0.941 ±0.025 | 1.0 | 0.865 | 0.34 | 0.053 | 0.294 | 1.05 |
| crag | 0.942 ±0.043 | 0.489 ±0.149 | 0.764 ±0.055 | 0.606 ±0.106 | 0.893 ±0.029 | 1.0 | 0.885 | 0.426 | 0.007 | 0.059 | 1.01 |
| relevance_gate | 0.942 ±0.043 | 0.489 ±0.149 | 0.635 ±0.075 | 0.521 ±0.117 | 0.927 ±0.03 | 1.0 | 0.779 | 0.426 | 0.212 | 0.941 | 0.8 |
| default_qwen3-0.6b | 0.962 ±0.034 | 0.489 ±0.149 | 0.649 ±0.062 | 0.5 ±0.08 | 0.891 ±0.041 | 1.0 | 0.904 | 0.319 | 0.007 | 0.059 | 0.62 |
| dpo_answer | 0.942 ±0.043 | 0.489 ±0.149 | 0.913 ±0.038 | 0.638 ±0.096 | 0.983 ±0.01 | 1.0 | 0.942 | 0.447 | 0.0 | 0.0 | 1.48 |
| grpo_answer | 0.942 ±0.043 | 0.489 ±0.149 | 0.817 ±0.055 | 0.426 ±0.106 | 0.981 ±0.016 | 1.0 | 0.885 | 0.298 | 0.04 | 0.118 | 1.02 |
| grpo_decompose | 0.923 ±0.048 | 0.447 ±0.138 | 0.788 ±0.053 | 0.617 ±0.096 | 0.883 ±0.033 | 1.0 | 0.894 | 0.383 | 0.0 | 0.059 | 0.98 |
| dpo_gate | 0.942 ±0.043 | 0.489 ±0.149 | 0.75 ±0.072 | 0.553 ±0.117 | 0.973 ±0.015 | 1.0 | 0.788 | 0.404 | 0.212 | 0.941 | 1.21 |
| dpo_refusal | 0.942 ±0.043 | 0.489 ±0.149 | 0.827 ±0.053 | 0.543 ±0.101 | 0.944 ±0.023 | 1.0 | 0.904 | 0.362 | 0.079 | 1.0 | 1.0 |
| dpo_refusal_gate | 0.942 ±0.043 | 0.489 ±0.149 | 0.712 ±0.075 | 0.5 ±0.122 | 0.963 ±0.021 | 1.0 | 0.788 | 0.362 | 0.219 | 1.0 | 0.86 |
| late_interaction | 0.962 ±0.034 | 0.447 ±0.149 | 0.755 ±0.058 | 0.606 ±0.101 | 0.892 ±0.028 | 1.0 | 0.923 | 0.404 | 0.007 | 0.059 | 0.97 |
| iterative_retrieval | 0.952 ±0.043 | 0.468 ±0.149 | 0.774 ±0.055 | 0.564 ±0.096 | 0.881 ±0.038 | 1.0 | 0.942 | 0.426 | 0.007 | 0.118 | 1.45 |
| context_window | 0.942 ±0.043 | 0.489 ±0.149 | 0.764 ±0.058 | 0.532 ±0.106 | 0.838 ±0.044 | 1.0 | 0.865 | 0.404 | 0.013 | 0.059 | 1.01 |
| read_more_sections | 0.942 ±0.043 | 0.489 ±0.149 | 0.774 ±0.06 | 0.511 ±0.101 | 0.904 ±0.032 | 1.0 | 0.885 | 0.426 | 0.013 | 0.059 | 0.95 |
| compressed_context | 0.942 ±0.043 | 0.489 ±0.149 | 0.745 ±0.053 | 0.606 ±0.096 | 0.9 ±0.031 | 1.0 | 0.923 | 0.404 | 0.007 | 0.059 | 0.96 |


In [11]:
release_llms()